# Guía 05 · Algoritmos de extracción de características para RA (SIFT · SURF · ORB)
**Unidad:** Interfaz Hombre-Máquina · **Laboratorio:** Diseño de Interfaces · **Proyecto:** PC-Builder AR

**Objetivo.** Ejecutar tres algoritmos de reconocimiento de objetos de OpenCV sobre *su propia imagen*,
explicar qué hace cada línea y compararlos (keypoints, descriptor, tiempo, calidad del emparejamiento).

**Cómo usarlo en Google Colab**
1. Suba **su imagen** (por ejemplo, una foto de un componente de PC: CPU, RAM, GPU o placa base). *Debe cambiar la imagen de ejemplo.*
2. Ejecute las celdas en orden (`Entorno ▸ Ejecutar todo`).
3. Copie las tablas y figuras al informe y escriba sus conclusiones al final.

> **Relación con la app web.** AR.js (usado en PC-Builder AR) detecta un *marcador* (patrón Hiro) con ARToolKit.
> Los algoritmos de este cuaderno son la base del **seguimiento de imágenes naturales (NFT)**: reconocen un objeto
> sin marcador usando puntos clave y descriptores. Aquí se comparan para entender cuándo conviene cada enfoque.

## 1. Entorno

In [ ]:
import os, time
import cv2                      # OpenCV: biblioteca de visión por computador
import numpy as np              # arreglos numéricos (una imagen es una matriz de píxeles)
import matplotlib.pyplot as plt # para mostrar las imágenes
import pandas as pd             # para la tabla comparativa
from IPython.display import display

print("OpenCV", cv2.__version__)

## 2. Cargar **su** imagen
Una imagen digital es una matriz de píxeles; en color cada píxel tiene 3 valores (B, G, R en OpenCV) y en
escala de grises uno solo (0 = negro … 255 = blanco).

In [ ]:
IMG_PATH = "mi_imagen.jpg"      # <-- CAMBIE ESTE NOMBRE por el de su imagen (no use la de ejemplo de otros)

try:                            # En Colab: abre el selector de archivos para subir la imagen
    from google.colab import files
    if not os.path.exists(IMG_PATH):
        subido = files.upload()
        IMG_PATH = next(iter(subido))
except ImportError:             # Fuera de Colab: se usa la ruta tal cual
    pass

img_color = cv2.imread(IMG_PATH)                  # lee el archivo -> matriz (alto, ancho, 3) en BGR
assert img_color is not None, f"No se pudo leer {IMG_PATH}"
img_color = cv2.resize(img_color, (640, int(640 * img_color.shape[0] / img_color.shape[1])))  # ancho fijo 640 px: tiempos comparables
gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)  # color -> grises: menos datos, algoritmo más simple y rápido

print("Color :", img_color.shape, "->", img_color.shape[0] * img_color.shape[1], "píxeles x 3 canales")
print("Grises:", gray.shape)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)); plt.title("Original (color)"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(gray, cmap="gray"); plt.title("Escala de grises"); plt.axis("off")
plt.show()

## 3. Imagen “de escena” (lo que vería la cámara de la app)
Para evaluar el reconocimiento se necesita una **segunda vista del mismo objeto**. La simulamos rotando,
reduciendo la escala, cambiando la iluminación y añadiendo ruido: justo lo que pasa cuando el usuario mueve el celular.

In [ ]:
h, w = gray.shape
M = cv2.getRotationMatrix2D((w / 2, h / 2), 25, 0.75)              # rotar 25° y escalar al 75% alrededor del centro
escena_color = cv2.warpAffine(img_color, M, (w, h), borderValue=(127, 127, 127))  # aplica la transformación
escena_color = cv2.convertScaleAbs(escena_color, alpha=0.8, beta=25)              # cambia contraste/brillo (iluminación distinta)
rng = np.random.default_rng(0)
ruido = rng.normal(0, 6, escena_color.shape)                                        # ruido gaussiano de sensor
escena_color = np.clip(escena_color.astype(np.float32) + ruido, 0, 255).astype(np.uint8)
escena = cv2.cvtColor(escena_color, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(5, 4)); plt.imshow(escena, cmap="gray"); plt.title("Escena simulada"); plt.axis("off"); plt.show()

## 4. Los tres algoritmos, línea por línea

| | **SIFT** | **SURF** | **ORB** |
|---|---|---|---|
| Año / autores | 2004 · D. Lowe | 2006 · Bay, Tuytelaars, Van Gool | 2011 · Rublee, Rabaud, Konolige, Bradski |
| Detector | Diferencia de Gaussianas (DoG) + serie de Taylor | Matriz Hessiana sobre **imagen integral** (filtros de caja) | **FAST** + medida de esquinas de Harris + pirámide |
| Descriptor | Histograma de gradientes: 16 vecinos × 8 bins = **128 números** (float) | Respuestas de wavelet: 16 vecinos × 4 = **64 números** (float) | **BRIEF** rotado: comparaciones de intensidad, **binario** (32 bytes) |
| Distancia al emparejar | L2 (euclídea) | L2 | **Hamming** (XOR + conteo de bits) |
| Licencia | libre (patente expirada, en OpenCV ≥ 4.4) | patentado: solo en *opencv-contrib* compilado con `NONFREE` | libre |

Cada algoritmo hace dos cosas: **detectar** puntos clave (*keypoints*) y **describirlos** (*descriptors*).

In [ ]:
def medir(detector, imagen, repeticiones=5):
    """Devuelve (keypoints, descriptores, tiempo medio en ms) de detectar+describir."""
    t = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        kp, des = detector.detectAndCompute(imagen, None)   # detecta keypoints y calcula su descriptor en un solo paso
        t.append((time.perf_counter() - t0) * 1000)          # ms
    return kp, des, float(np.mean(t))

# ---------- SIFT ----------
sift = cv2.SIFT_create()                  # crea el detector SIFT con parámetros por defecto
kp_s, des_s, t_s = medir(sift, gray)      # puntos clave + matriz (N x 128) de float32
kp_s2, des_s2, _ = medir(sift, escena)    # lo mismo sobre la escena simulada

# ---------- SURF (o respaldo) ----------
try:
    surf = cv2.xfeatures2d.SURF_create(hessianThreshold=400)  # umbral del determinante Hessiano: mayor = menos puntos
    NOMBRE_SURF, NORMA_SURF = "SURF", cv2.NORM_L2
except (AttributeError, cv2.error):
    # La mayoría de builds de OpenCV (Colab incluido) NO traen SURF por ser algoritmo patentado (módulo 'nonfree').
    surf = cv2.AKAZE_create() if hasattr(cv2, 'AKAZE_create') else cv2.xfeatures2d_AKAZE.create()   # sustituto libre, invariante a escala y rotación (4.x en Colab; 5.x cambia el nombre)
    NOMBRE_SURF, NORMA_SURF = "AKAZE (sustituye a SURF)", cv2.NORM_HAMMING
    print("SURF no está disponible en este OpenCV -> se usa AKAZE como sustituto y así queda indicado en la tabla.")
kp_f, des_f, t_f = medir(surf, gray)
kp_f2, des_f2, _ = medir(surf, escena)

# ---------- ORB ----------
orb = cv2.ORB_create(nfeatures=1500)      # FAST + Harris + BRIEF; limita a los 1500 mejores puntos
kp_o, des_o, t_o = medir(orb, gray)       # descriptor binario: matriz (N x 32) de uint8 = 256 bits
kp_o2, des_o2, _ = medir(orb, escena)

for nombre, kp, des in [("SIFT", kp_s, des_s), (NOMBRE_SURF, kp_f, des_f), ("ORB", kp_o, des_o)]:
    print(f"{nombre:26s} keypoints={len(kp):5d}  descriptor={des.shape}  tipo={des.dtype}")

### 4.1 Puntos clave encontrados
Cada círculo es un *keypoint*; el radio indica la **escala** y la línea la **orientación** (invariancia a escala y rotación).

In [ ]:
def dibujar(img_gray, kps, titulo, ax):
    vis = cv2.drawKeypoints(img_gray, kps, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)  # círculos con escala y ángulo
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.set_title(f"{titulo}: {len(kps)} puntos"); ax.axis("off")

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
dibujar(gray, kp_s, "SIFT", ax[0]); dibujar(gray, kp_f, NOMBRE_SURF, ax[1]); dibujar(gray, kp_o, "ORB", ax[2])
plt.tight_layout(); plt.show()

## 5. Coincidencia de características (*feature matching*)
Se comparan los descriptores de la imagen original con los de la escena:
1. `knnMatch(k=2)` busca los 2 vecinos más cercanos de cada descriptor.
2. **Prueba de razón de Lowe**: se acepta la pareja solo si el mejor vecino es claramente mejor que el segundo (`< 0.75`).
3. **Homografía + RANSAC**: de las parejas buenas se estima la transformación geométrica y se descartan las incoherentes (*outliers*).
   Los *inliers* que sobreviven son la evidencia real de que el objeto fue reconocido.

In [ ]:
def emparejar(kp1, des1, kp2, des2, norma, razon=0.75):
    t0 = time.perf_counter()
    bf = cv2.BFMatcher(norma)                                   # fuerza bruta: L2 para float, Hamming para binarios
    pares = bf.knnMatch(des1, des2, k=2)                        # 2 vecinos más cercanos por descriptor
    buenos = [p[0] for p in pares if len(p) == 2 and p[0].distance < razon * p[1].distance]   # prueba de razón de Lowe
    H, mascara, inliers = None, None, 0
    if len(buenos) >= 4:                                        # una homografía necesita al menos 4 parejas
        src = np.float32([kp1[m.queryIdx].pt for m in buenos]).reshape(-1, 1, 2)   # puntos en la imagen original
        dst = np.float32([kp2[m.trainIdx].pt for m in buenos]).reshape(-1, 1, 2)   # sus parejas en la escena
        H, mascara = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)  # RANSAC: tolera 5 px de error de reproyección
        inliers = int(mascara.sum()) if mascara is not None else 0
    return buenos, H, mascara, inliers, (time.perf_counter() - t0) * 1000

casos = {
    "SIFT": (kp_s, des_s, kp_s2, des_s2, cv2.NORM_L2),
    NOMBRE_SURF: (kp_f, des_f, kp_f2, des_f2, NORMA_SURF),
    "ORB": (kp_o, des_o, kp_o2, des_o2, cv2.NORM_HAMMING),
}
resultados, figuras = [], {}
tiempos = {"SIFT": t_s, NOMBRE_SURF: t_f, "ORB": t_o}
for nombre, (k1, d1, k2, d2, norma) in casos.items():
    buenos, H, mascara, inliers, t_match = emparejar(k1, d1, k2, d2, norma)
    figuras[nombre] = (k1, k2, buenos, H, mascara)
    resultados.append({
        "Algoritmo": nombre,
        "Keypoints (original)": len(k1),
        "Descriptor": f"{d1.shape[1]} x {d1.dtype}",
        "Tiempo detectar+describir (ms)": round(tiempos[nombre], 1),
        "Parejas buenas (Lowe)": len(buenos),
        "Inliers (RANSAC)": inliers,
        "% inliers": round(100 * inliers / len(buenos), 1) if buenos else 0.0,
        "Tiempo emparejar (ms)": round(t_match, 1),
        "¿Reconocido?": "Sí" if inliers >= 10 else "No",
    })
tabla = pd.DataFrame(resultados).set_index("Algoritmo")
display(tabla)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 14))
for a, (nombre, (k1, k2, buenos, H, mascara)) in zip(ax, figuras.items()):
    if H is not None:
        esq = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)    # esquinas de la imagen original
        proy = cv2.perspectiveTransform(esq, H)                                # dónde caen en la escena
        escena_vis = cv2.polylines(escena.copy(), [np.int32(proy)], True, 255, 3)   # contorno del objeto reconocido
    else:
        escena_vis = escena
    vis = cv2.drawMatches(gray, k1, escena_vis, k2, buenos, None,
                          matchesMask=mascara.ravel().tolist() if mascara is not None else None,   # solo se dibujan los inliers
                          flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    a.imshow(vis, cmap="gray"); a.set_title(f"{nombre}: líneas = inliers RANSAC"); a.axis("off")
plt.tight_layout(); plt.show()

## 6. Análisis y conclusiones (complete con **sus** resultados)

Use la tabla de la sección 5 y las figuras para responder en el informe:

1. **Cantidad de keypoints.** ¿Cuál detectó más puntos? ¿Más puntos significa mejor reconocimiento? Justifique con el *% de inliers*.
2. **Descriptor.** Compare el tamaño y tipo (128 float de SIFT, 64 float de SURF, 256 bits de ORB). ¿Cómo afecta a la memoria y a la distancia usada (L2 vs. Hamming)?
3. **Velocidad.** Ordene los algoritmos por tiempo. ¿Cuál sería viable a 30 fps en un celular de gama media, dentro de una app web de RA?
4. **Robustez.** La escena tiene rotación 25°, escala 0.75, otra iluminación y ruido. ¿Quién mantuvo más inliers?
5. **Elección para PC-Builder AR.** Argumente por qué la app usa un **marcador Hiro** (ARToolKit) y en qué casos usaría reconocimiento de imágenes naturales con alguno de estos algoritmos.

**Pistas de la teoría**
- SIFT es el más robusto y costoso; SURF acelera con imágenes integrales; ORB es binario, el más rápido y libre → típico en móviles.
- Un descriptor binario se compara con XOR + conteo de bits (Hamming), mucho más barato que una distancia euclídea sobre 128 *floats*.

**Escriba aquí sus conclusiones:**

*(…)*